In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import string
import numpy as np

In [2]:
with open('girls-names.csv') as f:
    content = f.readlines()
print(len(content))
content = ' '.join( [c.strip()+'.' for c in content])
content[:100]

3759


'Aadhya. Aadya. Aaira. Aairah. Aaliyah. Aanya. Aaradhya. Aarna. Aarvi. Aarya. Aashvi. Aavya. Aayat. A'

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)

mps


In [4]:
allowed_chars = set(string.ascii_letters + string.digits + ". ")

content = ''.join([c for c in content if c in allowed_chars])

chars = set(sorted(list(content)))
i_to_c = {k:v for k, v in enumerate(chars)}
c_to_i = {k:v for v, k in i_to_c.items()}

encode = lambda s: [c_to_i[c] for c in s] # noqa: E731
decode = lambda s: ''.join([i_to_c[i] for i in s]) # noqa: E731
decode(encode(content[:100]))

encoded_content = torch.tensor(encode(content))

In [5]:
print(chars, len(chars))

{'y', 'w', '.', 'f', 'U', 'q', 'W', 'd', 'P', 'e', 'k', 'F', 'Z', 'A', 'u', 'E', 'r', 'Q', 'L', 'z', 'c', 'Y', 't', 'K', ' ', 'g', 'v', 'O', 'X', 'V', 'T', 'm', 'N', 'p', 'n', 'h', 'B', 'G', 'R', 'l', 'o', 's', 'j', 'b', 'D', 'C', 'a', 'i', 'I', 'x', 'S', 'H', 'M', 'J'} 54


In [6]:
blocks = []
block_length = 3
batch_size = 32

xs = []
ys = []

for i in range(0, len(encoded_content)-block_length):
    x_chunk = encoded_content[i:i+block_length-1]
    y_chunk = encoded_content[i+block_length]

    xs.append(torch.tensor(x_chunk))
    ys.append(torch.tensor(y_chunk))

X = F.one_hot(torch.stack(xs)).float()
Y = torch.stack(ys)


/var/folders/x9/xbf96x5928q0w6kkxyy3cvyh0000gp/T/ipykernel_57960/1668716928.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs.append(torch.tensor(x_chunk))
/var/folders/x9/xbf96x5928q0w6kkxyy3cvyh0000gp/T/ipykernel_57960/1668716928.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  ys.append(torch.tensor(y_chunk))


In [7]:
X.shape

torch.Size([30057, 2, 54])

In [8]:
X = X.to(device)
Y = Y.to(device)

In [9]:

hidden_layer_width = 512
relu = torch.nn.ReLU()

W1 = torch.randn(len(chars)*2, hidden_layer_width, device=device)*0.01
W1.requires_grad_(True)

b1 = torch.zeros(hidden_layer_width, requires_grad=True, device=device)

W2 = torch.randn(hidden_layer_width, len(chars), device=device)*0.01
W2.requires_grad_(True)

b2 = torch.zeros(len(chars), requires_grad=True, device=device)

In [10]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter()

In [11]:
for epoch in range(10):

    indices = torch.randperm(len(X))
    indices = indices.to(device)

    for i in range(0, len(X), batch_size):
        
        batch_idx = indices[i:i+batch_size]
        
        x = X[batch_idx].view(len(batch_idx), -1)
        
        y = Y[batch_idx]
        z1 = x @ W1 + b1
        h = relu(z1)
        logits = h @ W2 + b2
        
        loss = F.cross_entropy(logits.view(-1, len(chars)), y.view(-1))
        writer.add_scalar("Loss/train", loss, i)

        if W1.grad is not None:
            for p in [W1, W2, b1, b2]: 
                p.grad.zero_()

        loss.backward()
    
        with torch.no_grad():
            for p in [W1, W2, b1, b2]:
                p -= 0.01 * p.grad
        
        if not i%1000:
            print(f'epoch {epoch}, loss {loss}, {i} of {len(X)} iterations')
            
        

epoch 0, loss 3.987900972366333, 0 of 30057 iterations
epoch 0, loss 3.9265944957733154, 4000 of 30057 iterations
epoch 0, loss 3.8653035163879395, 8000 of 30057 iterations
epoch 0, loss 3.8387036323547363, 12000 of 30057 iterations
epoch 0, loss 3.741062641143799, 16000 of 30057 iterations
epoch 0, loss 3.677824020385742, 20000 of 30057 iterations
epoch 0, loss 3.586697816848755, 24000 of 30057 iterations
epoch 0, loss 3.681635856628418, 28000 of 30057 iterations
epoch 1, loss 3.5276002883911133, 0 of 30057 iterations
epoch 1, loss 3.4641988277435303, 4000 of 30057 iterations
epoch 1, loss 3.3109443187713623, 8000 of 30057 iterations
epoch 1, loss 3.358280658721924, 12000 of 30057 iterations
epoch 1, loss 3.469451427459717, 16000 of 30057 iterations
epoch 1, loss 3.2041587829589844, 20000 of 30057 iterations
epoch 1, loss 3.058419704437256, 24000 of 30057 iterations
epoch 1, loss 2.7486801147460938, 28000 of 30057 iterations
epoch 2, loss 3.1613504886627197, 0 of 30057 iterations
epoc

In [12]:
inputs = ' B'

for i in range(100):
    x = F.one_hot(torch.tensor(encode(inputs[-2:]), device=device), num_classes=len(chars))

    z1 = x.float().view(-1,) @ W1 + b1
    relu = torch.nn.ReLU()
    h = relu(z1)
    logits = h @ W2 + b2
    
    probs = F.softmax(logits, dim=-1)
    idx = torch.multinomial(probs, num_samples=1).item()
    inputs+=i_to_c[idx]

    print(inputs)



 Bi
 Bie
 Bie 
 Bie h
 Bie hr
 Bie hra
 Bie hra 
 Bie hra .
 Bie hra .S
 Bie hra .Sr
 Bie hra .Sr.
 Bie hra .Sr.d
 Bie hra .Sr.da
 Bie hra .Sr.dad
 Bie hra .Sr.dad.
 Bie hra .Sr.dad.E
 Bie hra .Sr.dad.Ea
 Bie hra .Sr.dad.Eas
 Bie hra .Sr.dad.Easi
 Bie hra .Sr.dad.Easi.
 Bie hra .Sr.dad.Easi.a
 Bie hra .Sr.dad.Easi.aa
 Bie hra .Sr.dad.Easi.aai
 Bie hra .Sr.dad.Easi.aaig
 Bie hra .Sr.dad.Easi.aaig.
 Bie hra .Sr.dad.Easi.aaig..
 Bie hra .Sr.dad.Easi.aaig..e
 Bie hra .Sr.dad.Easi.aaig..el
 Bie hra .Sr.dad.Easi.aaig..ele
 Bie hra .Sr.dad.Easi.aaig..elel
 Bie hra .Sr.dad.Easi.aaig..elely
 Bie hra .Sr.dad.Easi.aaig..elely.
 Bie hra .Sr.dad.Easi.aaig..elely.A
 Bie hra .Sr.dad.Easi.aaig..elely.An
 Bie hra .Sr.dad.Easi.aaig..elely.Ann
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.X
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.Xg
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.Xgt
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.Xgt.
 Bie hra .Sr.dad.Easi.aaig..elely.Ann.Xgt..
 Bie hra .Sr.dad

In [99]:
import torch.nn as nn

context_length = 2

class NameGenerator(nn.Module):
    def __init__(self, vocab_size, context_length, hidden_dim):
        super().__init__()
        
        input_dim = context_length * vocab_size

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
            )

    def forward(self, x):
        
        return self.net(x)

model = NameGenerator(len(chars), context_length, 512)
optimizer = torch.optim.Adam(model.parameters(), lr=.01)

In [38]:
encoded_content.shape, context_length

(torch.Size([30060]), 2)

In [96]:
from torch.utils.data import TensorDataset, DataLoader


xs = []
ys = []
for i in range(0, len(encoded_content)-context_length):
    x_chunk = encoded_content[i:i+context_length]
    y_chunk = encoded_content[i+context_length]

    xs.append(x_chunk)
    ys.append(y_chunk)

X = torch.stack(xs).float()
Y = torch.stack(ys)

dataset = TensorDataset(X, Y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [101]:
for xb, yb in loader:
    print(xb.shape)
    break

torch.Size([32, 2])


In [100]:
for xb, yb in loader:
    logits = model.forward(xb)
    loss = F.cross_entropy(logits.view(-1, len(chars)), yb)
    writer.add_scalar("Loss/train", loss, i)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x2 and 108x512)